# 02 — Single-Point CIR Sanity Test

Place anchor_0 as TX and room centre as RX.
Run Sionna RT to compute multipath CIR.
Assert: first-path range ≈ Euclidean distance to within 1 cm (confirms LOS and correct coordinate frame).

In [ ]:
import os
import numpy as np
import yaml
import matplotlib.pyplot as plt
import sionna
import sionna.rt as rt
from sionna.rt import (PathSolver, Transmitter, Receiver,
                       PlanarArray, Camera, InteractionType)

print(f"Sionna {sionna.__version__}")

## Load scene and set frequency

In [ ]:
scene_path = os.path.abspath("../scene/room.xml")
scene = rt.load_scene(scene_path)
scene.frequency = 6.5e9          # DW3000 channel 5 centre frequency
print(f"Scene loaded  — frequency = {scene.frequency.item() / 1e9:.2f} GHz")
print(f"Wavelength    = {scene.wavelength.item() * 100:.2f} cm")

## Load anchor positions from config YAML

In [ ]:
anchors_yaml = os.path.abspath(
    "../../src/microuwb_bringup/config/anchors.yaml")
print(f"Reading: {anchors_yaml}")
with open(anchors_yaml, "r") as f:
    cfg = yaml.safe_load(f)

anchors = cfg["anchors"]
print("\nLoaded anchors:")
for a in anchors:
    print(f"  anchor_{a['id']}  ({a['x']}, {a['y']}, {a['z']})")

## Place TX (anchor_0) and RX (room centre)

In [ ]:
a0 = anchors[0]
TX_POS = [float(a0["x"]), float(a0["y"]), float(a0["z"])]
RX_POS = [2.5, 2.0, 1.0]    # room centre at 1 m height

euclidean_dist = float(np.linalg.norm(np.array(TX_POS) - np.array(RX_POS)))

print(f"TX anchor_0 position : {TX_POS}")
print(f"RX room-centre       : {RX_POS}")
print(f"Euclidean distance   : {euclidean_dist:.6f} m")

## Configure antennas and add devices

In [ ]:
# Isotropic single-element arrays (DW3000 approximation)
iso_array = PlanarArray(num_rows=1, num_cols=1,
                        vertical_spacing=0.5, horizontal_spacing=0.5,
                        pattern="iso", polarization="V")
scene.tx_array = iso_array
scene.rx_array = iso_array

scene.add(Transmitter(name="tx", position=TX_POS))
scene.add(Receiver(name="rx",   position=RX_POS))
print("TX and RX added")

## Compute paths

`max_depth=4` allows up to 4 reflections.
`refraction=False` keeps walls opaque (appropriate for concrete at 6.5 GHz).
`diffraction=True` captures edge effects around doorframes/corners in future scenes.

In [ ]:
solver = PathSolver()
paths = solver(
    scene,
    max_depth=4,
    los=True,
    specular_reflection=True,
    diffuse_reflection=False,
    refraction=False,
    diffraction=True,
    samples_per_src=int(1e6),
)
print(f"tau shape          : {paths.tau.shape}")
print(f"valid shape        : {paths.valid.shape}")
print(f"interactions shape : {paths.interactions.shape}")

## Extract first-path delay and compute range

In [ ]:
# Convert to numpy and collapse singleton TX/RX/antenna dimensions
# tau   shape: (num_rx, num_tx, num_paths)       with synthetic_array=True
# valid shape: (num_rx, num_rx_ant, num_tx, num_tx_ant, num_paths) always
tau_all   = np.array(paths.tau).squeeze()     # → (num_paths,)
valid_all = np.array(paths.valid).squeeze()   # → (num_paths,)
valid_mask = valid_all.astype(bool)

tau_valid  = tau_all[valid_mask]
num_valid  = int(valid_mask.sum())

print(f"Total path slots : {len(tau_all)}")
print(f"Valid paths      : {num_valid}")

if num_valid == 0:
    raise RuntimeError(
        "No valid paths found.  Check that TX/RX are not inside walls "
        "and that scene normals point into the room.")

first_idx     = int(np.argmin(tau_valid))
first_delay_s = float(tau_valid[first_idx])

SPEED_OF_LIGHT = 3e8  # m/s
first_delay_ns  = first_delay_s * 1e9
first_range_m   = first_delay_s * SPEED_OF_LIGHT
range_error_cm  = abs(first_range_m - euclidean_dist) * 100

print()
print(f"First-path delay   : {first_delay_ns:.4f} ns")
print(f"First-path range   : {first_range_m:.4f} m")
print(f"Euclidean distance : {euclidean_dist:.4f} m")
print(f"Range error        : {range_error_cm:.3f} cm")

## LOS detection and sanity assertion

In [ ]:
# interactions shape: (max_depth, num_rx[_ant], num_tx[_ant], num_paths)
# Collapse everything except axis-0 and axis-(-1)
inter = np.array(paths.interactions)
md_  = inter.shape[0]
np_  = inter.shape[-1]
inter_2d = inter.reshape(md_, -1, np_)[:, 0, :]      # (max_depth, num_paths)
inter_valid = inter_2d[:, valid_mask]                  # (max_depth, num_valid_paths)
first_path_inter = inter_valid[:, first_idx]           # (max_depth,)

# LOS path has NONE (=0) at every depth
los_flag = bool(np.all(first_path_inter == InteractionType.NONE))

print(f"First-path interactions : {first_path_inter.tolist()}")
print(f"LOS detected            : {los_flag}")

TOLERANCE_M = 0.01   # 1 cm
err_m = abs(first_range_m - euclidean_dist)

print()
if los_flag and err_m < TOLERANCE_M:
    print("SANITY CHECK PASSED")
    print(f"  LOS range error {err_m * 100:.3f} cm is within {TOLERANCE_M * 100:.0f} cm tolerance")
else:
    msg_parts = []
    if not los_flag:
        msg_parts.append(
            "First path is NOT classified as LOS — possible wrong surface normals "
            "(check gen_meshes.py winding) or TX/RX inside a wall.")
    if err_m >= TOLERANCE_M:
        msg_parts.append(
            f"Range error {err_m:.4f} m exceeds {TOLERANCE_M} m tolerance — "
            "possible coordinate-frame mismatch between scene XML and anchors.yaml.")
    for msg in msg_parts:
        print(f"FAIL: {msg}")
    raise AssertionError("\n".join(msg_parts))

## Path visualisation

In [ ]:
import os
RENDERS = os.path.abspath("../renders")
os.makedirs(RENDERS, exist_ok=True)

cam = Camera(position=[-3.0, -2.0, 5.0], look_at=[2.5, 2.0, 1.5])
bmp = scene.render(camera=cam, paths=paths, num_samples=128,
                   resolution=(720, 540), show_devices=True,
                   return_bitmap=True)
img = np.array(bmp)
fig, ax = plt.subplots(figsize=(9, 6))
ax.imshow(img); ax.set_title("Multipath from anchor_0 to room centre"); ax.axis("off")
plt.tight_layout()
out = os.path.join(RENDERS, "02_paths.png")
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.close()
print(f"Saved: {out}")

## Summary

| Metric | Value |
|---|---|
| TX (anchor_0) | (0.1, 0.1, 2.5) |
| RX (room centre) | (2.5, 2.0, 1.0) |
| Frequency | 6.5 GHz |
| First-path range | ≈ Euclidean distance |
| LOS | True |

Step 3a-ii will run this computation over a 2D grid and save to HDF5.